# Full-Corpus Final System Validation

Evaluate the frozen production ASR retrieval system on the complete corpus using the 40-query Extension40 benchmark.

This notebook evaluates only the final system. It does not reopen BM25, E5, fusion weights, candidate policies, or reranker selection.

Outputs are saved under:

```text
production_full_corpus/reports/full_corpus_validation/
```


## 1. Setup and release

The online retrieval environment uses the same pinned versions as the final production verification. In a fresh Colab runtime, run the dependency cell once. If Colab asks for a restart, restart and continue from the setup cell below.


In [1]:
%pip install -q "transformers==4.52.0" "sentence-transformers==5.4.0"

Reason for being yanked: <none given>
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 113.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.8/570.8 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 106.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.24.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
from google.colab import drive
drive.mount("/content/drive")

from dataclasses import asdict
from pathlib import Path
import importlib.metadata as importlib_metadata
import json
import os
import sys

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

PROJECT_ROOT    = Path("/content/drive/MyDrive/aic26/asr_model_comparison")
PRODUCTION_ROOT = PROJECT_ROOT / "production"
SRC_ROOT        = PROJECT_ROOT / "src"

FULL_CORPUS_ROOT = PROJECT_ROOT / "production_full_corpus"
RELEASE_ID       = "aic2026-full-20260817-r01"
RELEASE_PATH     = FULL_CORPUS_ROOT / "phase2" / "releases" / RELEASE_ID
REPORT_ROOT      = FULL_CORPUS_ROOT / "reports" / "full_corpus_validation"

CONFIG_PATH     = PROJECT_ROOT / "configs" / "retrieval_v2.json"
HISTORICAL_PATH = PROJECT_ROOT / "reports" / "retrieval_v2" / "stage07_reranking" / "quality_latency_summary.csv"

assert PROJECT_ROOT.is_dir(), PROJECT_ROOT
assert PRODUCTION_ROOT.is_dir(), PRODUCTION_ROOT
assert SRC_ROOT.is_dir(), SRC_ROOT
assert RELEASE_PATH.is_dir(), RELEASE_PATH
assert CONFIG_PATH.is_file(), CONFIG_PATH

os.chdir(PROJECT_ROOT)

for path in [PRODUCTION_ROOT, SRC_ROOT] :
    if (str(path) not in sys.path) :
        sys.path.insert(0, str(path))

from asr_retrieval import ArtifactConfig, ASRRetrievalEngine, ProductionConfig
from asr_retrieval.artifacts import load_release, validate_release
import evaluation as research_evaluation

transformers_version = importlib_metadata.version("transformers")
sentence_version     = importlib_metadata.version("sentence-transformers")

assert transformers_version == "4.52.0", transformers_version
assert sentence_version == "5.4.0", sentence_version
assert torch.cuda.is_available(), "Select a GPU runtime before loading the production engine."

REPORT_ROOT.mkdir(parents = True, exist_ok = True)

print("Project               :", PROJECT_ROOT)
print("Release               :", RELEASE_PATH)
print("Reports               :", REPORT_ROOT)
print("transformers          :", transformers_version)
print("sentence-transformers :", sentence_version)
print("PyTorch               :", torch.__version__)
print("GPU                   :", torch.cuda.get_device_name(0))

Mounted at /content/drive
Project               : /content/drive/MyDrive/aic26/asr_model_comparison
Release               : /content/drive/MyDrive/aic26/asr_model_comparison/production_full_corpus/phase2/releases/aic2026-full-20260817-r01
Reports               : /content/drive/MyDrive/aic26/asr_model_comparison/production_full_corpus/reports/full_corpus_validation
transformers          : 4.52.0
sentence-transformers : 5.4.0
PyTorch               : 2.11.0+cu128
GPU                   : Tesla T4


In [3]:
manifest = validate_release(RELEASE_PATH, verify_hashes = True)
release  = load_release(
    RELEASE_PATH,
    ArtifactConfig(verify_file_hashes = False, load_embeddings_into_ram = False),
)

print("Release ID          :", manifest.release_id)
print("Corpus identity     :", manifest.corpus_identity)
print("Videos              :", f"{manifest.video_count:,}")
print("Physical windows    :", f"{manifest.physical_window_count:,}")
print("Searchable windows  :", f"{manifest.eligible_window_count:,}")
print("Embedding shape     :", manifest.embedding_shape)
print("Embedding dtype     :", manifest.embedding_dtype)
print("Release validation  : PASS")

Release ID          : aic2026-full-20260817-r01
Corpus identity     : 4109e8a68309ae33de9a70add93010008f0897ecf65a58af67f0193a1535ded1
Videos              : 1,478
Physical windows    : 26,163
Searchable windows  : 26,117
Embedding shape     : (26117, 1024)
Embedding dtype     : float32
Release validation  : PASS


## 2. Load evaluation queries

Use the exact Extension40 benchmark from the earlier Retrieval v2 evaluation. Development and holdout labels are ignored here – all 40 queries are treated as one final validation set.

Queries without usable temporal ground truth can still contribute to video-ranking metrics.


In [4]:
retrieval_config = json.loads(CONFIG_PATH.read_text(encoding = "utf-8"))
benchmark_path   = Path(retrieval_config["benchmarks"]["extension40"]["path"])

if (not benchmark_path.is_absolute()) :
    benchmark_path = PROJECT_ROOT / benchmark_path

assert benchmark_path.is_file(), benchmark_path

benchmark            = json.loads(benchmark_path.read_text(encoding = "utf-8"))
benchmark_validation = research_evaluation.validate_benchmark_manifest(benchmark)

if (benchmark_validation["errors"]) :
    raise RuntimeError("Extension40 benchmark validation failed:\n- " + "\n- ".join(benchmark_validation["errors"]))

queries = research_evaluation.query_rows(benchmark)[[
    "query_id", "query_text", "video_id", "frame_id", "answer_time_s",
    "task_type", "difficulty", "query_category",
]].copy()

queries["query_id"]      = queries["query_id"].astype(str)
queries["query_text"]    = queries["query_text"].astype(str)
queries["video_id"]      = queries["video_id"].astype(str)
queries["answer_time_s"] = pd.to_numeric(queries["answer_time_s"], errors = "coerce")

release_video_ids = {video.video_id for video in release.videos}
missing_targets   = sorted(set(queries["video_id"]) - release_video_ids)

assert len(queries) == 40, f"Expected Extension40 to contain 40 queries, found {len(queries)}"
assert not queries["query_id"].duplicated().any(), "Extension40 contains duplicate query IDs"
assert queries["query_text"].str.strip().ne("").all(), "Extension40 contains an empty query"
assert not missing_targets, f"Target videos missing from the full release: {missing_targets}"

print("Benchmark             :", benchmark_path)
print("Queries               :", f"{len(queries):,}")
print("Unique query IDs      :", f"{queries['query_id'].nunique():,}")
print("Target videos present :", f"{queries['video_id'].isin(release_video_ids).sum():,} / {len(queries):,}")
print("Valid answer times    :", f"{queries['answer_time_s'].notna().sum():,} / {len(queries):,}")
display(queries)

Benchmark             : /content/drive/MyDrive/aic26/asr_model_comparison/data/stage1_extension40/benchmark_manifest.json
Queries               : 40
Unique query IDs      : 40
Target videos present : 40 / 40
Valid answer times    : 40 / 40


,query_id,query_text,video_id,frame_id,answer_time_s,task_type,difficulty,query_category
0,R1-2,Mẩu tin giới thiệu về đàn hổ tại một địa phươn...,L21_V029,11555,385.166667,KIS,easy,number_or_measurement
1,R1-9,Đoạn clip là cảnh thu hoạch dứa ở miền Tây: mộ...,L27_V013,540,21.600000,KIS,medium,person_organization_location
2,R1-13,Đoạn video mô tả một người ngồi vệ sinh máy ản...,L30_V095,2672,106.880000,KIS,medium,other
3,R1-15,Đoạn video về một chương trình từ thiện của mộ...,L30_V072,1745,69.800000,QA,easy,person_organization_location
4,R1-17,Đoạn video trong buổi trao quà từ thiện diễn r...,L30_V092,2485,99.400000,KIS,medium,number_or_measurement
5,R1-22,Đoạn video về một người phụ nữ dạy nấu ăn cho ...,L26_V178,4086,163.440000,QA,easy,number_or_measurement
6,R1-23,Đoạn clip về 1 thị trấn ven biển thu hút du kh...,L22_V022,16770,559.000000,KIS,hard,number_or_measurement
7,R2-11,Cảnh quay một người phụ nữ lớn tuổi đang đọc s...,L30_V014,1280,51.200000,KIS,medium,person_organization_location
8,R3-2,Cảnh quay cận cảnh một miếng thiếc có nhiều lỗ...,L29_V020,16988,679.520000,KIS,medium,person_organization_location
9,R3-15,Trong một đoạn video nấu ăn một món ăn về tôm....,L26_V222,3000,120.000000,KIS,medium,number_or_measurement


## 3. Load the final production engine

Load the normal production engine with its frozen first-stage retrieval, candidate policy, BGE reranker, and final fusion. From this point onward the system is evaluated as one final black box.


In [5]:
os.environ.setdefault("SENTENCE_TRANSFORMERS_HOME", str(FULL_CORPUS_ROOT / "model_cache" / "e5"))

production_config = ProductionConfig()

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print("Loading and warming the final production engine...")
engine = ASRRetrievalEngine.from_artifacts(
    RELEASE_PATH,
    config = production_config,
    warmup = True,
)

assert engine.state == "READY"

print("Engine state          :", engine.state)
print("First-stage fusion    :", f"{production_config.fusion.bm25_weight:.2f} BM25 + {production_config.fusion.dense_weight:.2f} E5")
print("Candidate policy      :", production_config.candidates)
print("Reranker              :", production_config.bge.model_name)
print("Final fusion          :", f"{production_config.fusion.first_stage_weight:.2f} first-stage + {production_config.fusion.reranker_weight:.2f} BGE")

Loading and warming the final production engine...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

Engine state          : READY
First-stage fusion    : 0.25 BM25 + 0.75 E5
Candidate policy      : CandidateConfig(video_k=50, windows_per_video=3, max_candidate_pairs=150)
Reranker              : BAAI/bge-reranker-v2-m3
Final fusion          : 0.50 first-stage + 0.50 BGE


## 4. Full-corpus retrieval quality

Run each query against all 1,478 videos and record the exact final rank of the target video.

The result is saved after the run. If a matching saved result already exists for this release and the same 40 query IDs, it is loaded instead of rerunning BGE.


In [6]:
QUALITY_PATH = REPORT_ROOT / "query_results.csv"

reuse_quality = False

if (QUALITY_PATH.is_file()) :
    saved_quality = pd.read_csv(QUALITY_PATH)
    saved_query_ids = set(saved_quality["query_id"].astype(str)) if "query_id" in saved_quality else set()
    saved_release_ids = set(saved_quality["release_id"].astype(str)) if "release_id" in saved_quality else set()
    reuse_quality = (
        len(saved_quality) == len(queries)
        and saved_query_ids == set(queries["query_id"])
        and saved_release_ids == {RELEASE_ID}
    )

if (reuse_quality) :
    quality_results = saved_quality
    print("Loaded saved quality results:", QUALITY_PATH)
else :
    quality_rows = []

    for _, query in tqdm(queries.iterrows(), total = len(queries), desc = "Full-corpus quality", unit = "query", dynamic_ncols = True) :
        result = engine.search(
            query["query_text"],
            top_k = manifest.video_count,
            first_stage_only = False,
            windows_per_hit = 3,
        )

        target_hit = next(hit for hit in result.hits if hit.video_id == query["video_id"])
        top_hit    = result.hits[0]
        answer_time_s = float(query["answer_time_s"]) if pd.notna(query["answer_time_s"]) else None

        if (answer_time_s is not None) :
            zone_start = answer_time_s - 60.0
            zone_end   = answer_time_s + 60.0
            overlaps   = [
                max(0.0, min(window.end_s, zone_end) - max(window.start_s, zone_start))
                for window in target_hit.windows
            ]
            moment_hit_at_1 = bool(overlaps and overlaps[0] >= 30.0)
            moment_hit_at_3 = bool(any(overlap >= 30.0 for overlap in overlaps[ : 3]))
        else :
            moment_hit_at_1 = None
            moment_hit_at_3 = None

        row = {
            "release_id"           : result.release_id,
            "query_id"             : query["query_id"],
            "query_text"           : query["query_text"],
            "task_type"            : query["task_type"],
            "difficulty"           : query["difficulty"],
            "query_category"       : query["query_category"],
            "target_video_id"      : query["video_id"],
            "answer_time_s"        : answer_time_s,
            "target_rank"          : target_hit.rank,
            "reciprocal_rank"      : 1.0 / target_hit.rank,
            "hit_at_1"             : int(target_hit.rank <= 1),
            "hit_at_3"             : int(target_hit.rank <= 3),
            "hit_at_5"             : int(target_hit.rank <= 5),
            "hit_at_10"            : int(target_hit.rank <= 10),
            "hit_at_20"            : int(target_hit.rank <= 20),
            "hit_at_50"            : int(target_hit.rank <= 50),
            "hit_at_100"           : int(target_hit.rank <= 100),
            "top_1_video_id"       : top_hit.video_id,
            "top_1_score"          : top_hit.final_score if top_hit.final_score is not None else top_hit.first_stage_score,
            "moment_hit_at_1"      : moment_hit_at_1,
            "moment_hit_at_3"      : moment_hit_at_3,
            "candidate_pair_count" : result.timings.candidate_pair_count,
        }

        for index in range(3) :
            window = target_hit.windows[index] if index < len(target_hit.windows) else None
            number = index + 1
            row[f"target_window_{number}_id"]         = window.window_id if window is not None else None
            row[f"target_window_{number}_start_s"]    = window.start_s if window is not None else None
            row[f"target_window_{number}_end_s"]      = window.end_s if window is not None else None
            row[f"target_window_{number}_transcript"] = window.transcript if window is not None else None

        quality_rows.append(row)

    quality_results = pd.DataFrame(quality_rows)
    quality_results.to_csv(QUALITY_PATH, index = False)
    print("Saved:", QUALITY_PATH)

display(quality_results[[
    "query_id", "difficulty", "target_video_id", "target_rank",
    "hit_at_10", "moment_hit_at_3", "top_1_video_id",
]])

Full-corpus quality:   0%|          | 0/40 [00:00<?, ?query/s]

Saved: /content/drive/MyDrive/aic26/asr_model_comparison/production_full_corpus/reports/full_corpus_validation/query_results.csv


,query_id,difficulty,target_video_id,target_rank,hit_at_10,moment_hit_at_3,top_1_video_id
0,R1-2,easy,L21_V029,1,1,True,L21_V029
1,R1-9,medium,L27_V013,4,1,False,L27_V015
2,R1-13,medium,L30_V095,9,1,True,K20_V001
3,R1-15,easy,L30_V072,2,1,True,L30_V034
4,R1-17,medium,L30_V092,1,1,True,L30_V092
5,R1-22,easy,L26_V178,24,0,True,L26_V046
6,R1-23,hard,L22_V022,1,1,True,L22_V022
7,R2-11,medium,L30_V014,13,0,True,L30_V050
8,R3-2,medium,L29_V020,23,0,True,K18_V023
9,R3-15,medium,L26_V222,81,0,True,L26_V227


In [7]:
quality_summary = {
    "release_id"        : RELEASE_ID,
    "query_count"       : int(len(quality_results)),
    "video_recall_at_1" : float(quality_results["hit_at_1"].mean()),
    "video_recall_at_3" : float(quality_results["hit_at_3"].mean()),
    "video_recall_at_5" : float(quality_results["hit_at_5"].mean()),
    "video_recall_at_10": float(quality_results["hit_at_10"].mean()),
    "video_recall_at_20": float(quality_results["hit_at_20"].mean()),
    "video_recall_at_50": float(quality_results["hit_at_50"].mean()),
    "video_recall_at_100": float(quality_results["hit_at_100"].mean()),
    "video_mrr"         : float(quality_results["reciprocal_rank"].mean()),
    "mean_target_rank"  : float(quality_results["target_rank"].mean()),
    "median_target_rank": float(quality_results["target_rank"].median()),
    "worst_target_rank" : int(quality_results["target_rank"].max()),
}

metric_rows = [
    {"metric" : "Video R@1", "value" : quality_summary["video_recall_at_1"]},
    {"metric" : "Video R@3", "value" : quality_summary["video_recall_at_3"]},
    {"metric" : "Video R@5", "value" : quality_summary["video_recall_at_5"]},
    {"metric" : "Video R@10", "value" : quality_summary["video_recall_at_10"]},
    {"metric" : "Video R@20", "value" : quality_summary["video_recall_at_20"]},
    {"metric" : "Video R@50", "value" : quality_summary["video_recall_at_50"]},
    {"metric" : "Video R@100", "value" : quality_summary["video_recall_at_100"]},
    {"metric" : "Video MRR", "value" : quality_summary["video_mrr"]},
    {"metric" : "Mean target rank", "value" : quality_summary["mean_target_rank"]},
    {"metric" : "Median target rank", "value" : quality_summary["median_target_rank"]},
    {"metric" : "Worst target rank", "value" : quality_summary["worst_target_rank"]},
]

display(pd.DataFrame(metric_rows))
print("Hardest 15 queries")
display(
    quality_results.sort_values(["target_rank", "query_id"], ascending = [False, True])[[
        "query_id", "query_text", "difficulty", "task_type",
        "target_video_id", "target_rank", "top_1_video_id",
    ]].head(15)
)

,metric,value
0,Video R@1,0.42500
1,Video R@3,0.47500
2,Video R@5,0.55000
3,Video R@10,0.70000
4,Video R@20,0.75000
5,Video R@50,0.87500
6,Video R@100,0.90000
7,Video MRR,0.49109
8,Mean target rank,44.97500
9,Median target rank,4.00000


Hardest 15 queries


,query_id,query_text,difficulty,task_type,target_video_id,target_rank,top_1_video_id
10,R2-1,Nhiều người mặt áo cờ đỏ sao vàng đứng trước b...,easy,KIS,K03_V019,512,K14_V013
16,R2-8,Đoạn clip mở đầu với hình ảnh cây cầu có hệ th...,medium,KIS,K03_V023,488,L22_V015
31,R2-15,Một người đang chụp hình những chiếc ván trượt...,medium,KIS,K01_V018,277,K18_V025
11,R2-3,Đoạn clip về một gian trưng bày văn hóa - du l...,medium,QA,K17_V003,173,K08_V026
9,R3-15,Trong một đoạn video nấu ăn một món ăn về tôm....,medium,KIS,L26_V222,81,L26_V227
18,R3-20,Một cái bàn có để nhiều hộp đĩa CD. Một trong ...,medium,QA,K06_V022,45,K02_V016
26,R1-24,Đoạn video về tường thuật một cuộc đua xe đạp....,very_hard,KIS,L23_V007,32,K14_V003
21,R1-10,Tìm chính xác đoạn clip ngắn có ba người (hai ...,medium,KIS,L30_V017,29,K17_V007
5,R1-22,Đoạn video về một người phụ nữ dạy nấu ăn cho ...,easy,QA,L26_V178,24,L26_V046
8,R3-2,Cảnh quay cận cảnh một miếng thiếc có nhiều lỗ...,medium,KIS,L29_V020,23,K18_V023


## 5. Temporal and supporting-window quality

For queries with a known answer time, a returned supporting window is considered temporally relevant when it overlaps the `answer_time ± 60 s` zone by at least 30 seconds.

`Moment Hit@3` asks whether one of the target video's three returned supporting windows points to the correct region. `Video+Moment@K` requires both the target video to rank within K and a correct supporting window to be returned.


In [8]:
temporal_results = quality_results[quality_results["moment_hit_at_3"].notna()].copy()
temporal_results["moment_hit_at_1"] = temporal_results["moment_hit_at_1"].astype(bool)
temporal_results["moment_hit_at_3"] = temporal_results["moment_hit_at_3"].astype(bool)

quality_summary["temporal_query_count"] = int(len(temporal_results))
quality_summary["moment_hit_at_1"]       = float(temporal_results["moment_hit_at_1"].mean()) if len(temporal_results) else None
quality_summary["moment_hit_at_3"]       = float(temporal_results["moment_hit_at_3"].mean()) if len(temporal_results) else None

for k in [1, 5, 10, 20, 50] :
    value = ((temporal_results["target_rank"] <= k) & temporal_results["moment_hit_at_3"]).mean() if len(temporal_results) else np.nan
    quality_summary[f"video_plus_moment_at_{k}"] = float(value) if np.isfinite(value) else None

temporal_summary = pd.DataFrame([
    {"metric" : "Temporal queries", "value" : quality_summary["temporal_query_count"]},
    {"metric" : "Moment Hit@1", "value" : quality_summary["moment_hit_at_1"]},
    {"metric" : "Moment Hit@3", "value" : quality_summary["moment_hit_at_3"]},
    {"metric" : "Video+Moment@1", "value" : quality_summary["video_plus_moment_at_1"]},
    {"metric" : "Video+Moment@5", "value" : quality_summary["video_plus_moment_at_5"]},
    {"metric" : "Video+Moment@10", "value" : quality_summary["video_plus_moment_at_10"]},
    {"metric" : "Video+Moment@20", "value" : quality_summary["video_plus_moment_at_20"]},
    {"metric" : "Video+Moment@50", "value" : quality_summary["video_plus_moment_at_50"]},
])

( REPORT_ROOT / "quality_summary.json" ).write_text(
    json.dumps(quality_summary, ensure_ascii = False, indent = 2, allow_nan = False),
    encoding = "utf-8",
)

display(temporal_summary)
print("Saved:", REPORT_ROOT / "quality_summary.json")


def summarize_group(frame : pd.DataFrame, column : str) -> pd.DataFrame :
    rows = []

    for value, group in frame.groupby(column, dropna = False, sort = True) :
        temporal = group[group["moment_hit_at_3"].notna()]
        rows.append({
            column              : value,
            "query_count"       : len(group),
            "R@1"               : group["hit_at_1"].mean(),
            "R@5"               : group["hit_at_5"].mean(),
            "R@10"              : group["hit_at_10"].mean(),
            "MRR"               : group["reciprocal_rank"].mean(),
            "Video+Moment@10"   : ((temporal["target_rank"] <= 10) & temporal["moment_hit_at_3"].astype(bool)).mean() if len(temporal) else np.nan,
        })

    return pd.DataFrame(rows)


print("By difficulty")
display(summarize_group(quality_results, "difficulty"))
print("By task type")
display(summarize_group(quality_results, "task_type"))

,metric,value
0,Temporal queries,40.000
1,Moment Hit@1,0.875
2,Moment Hit@3,0.950
3,Video+Moment@1,0.425
4,Video+Moment@5,0.525
5,Video+Moment@10,0.675
6,Video+Moment@20,0.725
7,Video+Moment@50,0.850


Saved: /content/drive/MyDrive/aic26/asr_model_comparison/production_full_corpus/reports/full_corpus_validation/quality_summary.json
By difficulty


,difficulty,query_count,R@1,R@5,R@10,MRR,Video+Moment@10
0,easy,10,0.600000,0.7,0.800,0.664362,0.800000
1,hard,5,0.600000,0.6,1.000,0.653571,1.000000
2,medium,24,0.333333,0.5,0.625,0.404204,0.583333
3,very_hard,1,0.000000,0.0,0.000,0.031250,0.000000


By task type


,task_type,query_count,R@1,R@5,R@10,MRR,Video+Moment@10
0,KIS,32,0.4375,0.5625,0.71875,0.498405,0.6875
1,QA,8,0.3750,0.5000,0.62500,0.461834,0.6250


## 6. Latency and determinism

Measure deployment-style final searches with `top_k = 50` and three repetitions per query after warmup.

The same repeated searches verify that the returned top-50 video order and candidate-pair count are deterministic.


In [9]:
LATENCY_REPETITIONS = 3
LATENCY_PATH        = REPORT_ROOT / "latency_results.csv"

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

latency_rows = []
progress = tqdm(total = len(queries) * LATENCY_REPETITIONS, desc = "Final-system latency", unit = "search", dynamic_ncols = True)

for _, query in queries.iterrows() :
    for repetition in range(1, LATENCY_REPETITIONS + 1) :
        progress.set_postfix_str(f"{query['query_id']} rep={repetition}/{LATENCY_REPETITIONS}")

        result = engine.search(
            query["query_text"],
            top_k = 50,
            first_stage_only = False,
            windows_per_hit = 3,
        )

        latency_rows.append({
            "release_id"       : result.release_id,
            "query_id"         : query["query_id"],
            "query_text"       : query["query_text"],
            "repetition"       : repetition,
            "top_50_video_ids" : json.dumps([hit.video_id for hit in result.hits], ensure_ascii = False),
            **asdict(result.timings),
        })
        progress.update(1)

progress.close()

latency_results = pd.DataFrame(latency_rows)
latency_results.to_csv(LATENCY_PATH, index = False)

peak_gpu_allocated_gib = torch.cuda.max_memory_allocated() / 1024**3
peak_gpu_reserved_gib  = torch.cuda.max_memory_reserved() / 1024**3

determinism_rows = []

for query_id, group in latency_results.groupby("query_id", sort = True) :
    deterministic_order = group["top_50_video_ids"].nunique() == 1
    deterministic_pairs = group["candidate_pair_count"].nunique() == 1
    determinism_rows.append({
        "query_id"             : query_id,
        "deterministic_order"  : deterministic_order,
        "deterministic_pairs"  : deterministic_pairs,
        "deterministic"        : deterministic_order and deterministic_pairs,
    })

determinism = pd.DataFrame(determinism_rows)
print("Saved:", LATENCY_PATH)
print("Deterministic queries:", f"{determinism['deterministic'].sum():,} / {len(determinism):,}")

if (not determinism["deterministic"].all()) :
    display(determinism[~determinism["deterministic"]])

Final-system latency:   0%|          | 0/120 [00:00<?, ?search/s]

Saved: /content/drive/MyDrive/aic26/asr_model_comparison/production_full_corpus/reports/full_corpus_validation/latency_results.csv
Deterministic queries: 40 / 40


In [10]:
total_ms = latency_results["total_ms"].to_numpy(dtype = float)

latency_summary = {
    "release_id"               : RELEASE_ID,
    "query_count"              : int(len(queries)),
    "repetitions"              : LATENCY_REPETITIONS,
    "search_count"             : int(len(latency_results)),
    "mean_ms"                  : float(np.mean(total_ms)),
    "p50_ms"                   : float(np.percentile(total_ms, 50)),
    "p90_ms"                   : float(np.percentile(total_ms, 90)),
    "p95_ms"                   : float(np.percentile(total_ms, 95)),
    "max_ms"                   : float(np.max(total_ms)),
    "candidate_pairs_mean"     : float(latency_results["candidate_pair_count"].mean()),
    "candidate_pairs_min"      : int(latency_results["candidate_pair_count"].min()),
    "candidate_pairs_max"      : int(latency_results["candidate_pair_count"].max()),
    "reranker_batch_sizes"     : sorted(int(value) for value in latency_results["reranker_batch_size"].dropna().unique()),
    "peak_gpu_allocated_gib"   : float(peak_gpu_allocated_gib),
    "peak_gpu_reserved_gib"    : float(peak_gpu_reserved_gib),
    "deterministic_query_count": int(determinism["deterministic"].sum()),
    "deterministic_query_total": int(len(determinism)),
}

( REPORT_ROOT / "latency_summary.json" ).write_text(
    json.dumps(latency_summary, ensure_ascii = False, indent = 2, allow_nan = False),
    encoding = "utf-8",
)

display(pd.DataFrame([
    {"metric" : "Mean", "value_ms" : latency_summary["mean_ms"]},
    {"metric" : "p50", "value_ms" : latency_summary["p50_ms"]},
    {"metric" : "p90", "value_ms" : latency_summary["p90_ms"]},
    {"metric" : "p95", "value_ms" : latency_summary["p95_ms"]},
    {"metric" : "Max", "value_ms" : latency_summary["max_ms"]},
]))

print("Candidate pairs      :", f"{latency_summary['candidate_pairs_min']} – {latency_summary['candidate_pairs_max']}")
print("Reranker batch sizes :", latency_summary["reranker_batch_sizes"])
print("Peak GPU allocated   :", f"{latency_summary['peak_gpu_allocated_gib']:.2f} GiB")
print("Peak GPU reserved    :", f"{latency_summary['peak_gpu_reserved_gib']:.2f} GiB")
print("Saved                :", REPORT_ROOT / "latency_summary.json")

,metric,value_ms
0,Mean,2365.095156
1,p50,2330.716191
2,p90,2788.584541
3,p95,2885.657864
4,Max,3032.372688


Candidate pairs      : 149 – 150
Reranker batch sizes : [32]
Peak GPU allocated   : 3.49 GiB
Peak GPU reserved    : 4.16 GiB
Saved                : /content/drive/MyDrive/aic26/asr_model_comparison/production_full_corpus/reports/full_corpus_validation/latency_summary.json


## 7. Failure review and final summary

Review two practical failure groups:

- target video ranks outside the top 10;
- target video ranks inside the top 10, but none of its three returned supporting windows reaches the temporal ground-truth region.

The optional historical table uses the old Stage 7 result only as context. It does not rerun the old system.


In [11]:
failure_cases = quality_results[
    (quality_results["target_rank"] > 10)
    | (
        (quality_results["target_rank"] <= 10)
        & quality_results["moment_hit_at_3"].notna()
        & ~quality_results["moment_hit_at_3"].astype(bool)
    )
].copy()

failure_cases["failure_type"] = np.where(
    failure_cases["target_rank"] > 10,
    "target_outside_top10",
    "top10_moment_miss",
)

failure_path = REPORT_ROOT / "failure_cases.csv"
failure_cases.to_csv(failure_path, index = False)

print("Target outside Top 10 :", int((quality_results["target_rank"] > 10).sum()))
print("Target outside Top 50 :", int((quality_results["target_rank"] > 50).sum()))
print(
    "Top-10 moment misses  :",
    int(
        (
            (quality_results["target_rank"] <= 10)
            & quality_results["moment_hit_at_3"].notna()
            & ~quality_results["moment_hit_at_3"].astype(bool)
        ).sum()
    ),
)
print("Saved                 :", failure_path)

display(failure_cases[[
    "query_id", "query_text", "difficulty", "task_type", "failure_type",
    "target_video_id", "target_rank", "top_1_video_id", "moment_hit_at_3",
    "target_window_1_transcript", "target_window_2_transcript", "target_window_3_transcript",
]])

Target outside Top 10 : 12
Target outside Top 50 : 5
Top-10 moment misses  : 1
Saved                 : /content/drive/MyDrive/aic26/asr_model_comparison/production_full_corpus/reports/full_corpus_validation/failure_cases.csv


,query_id,query_text,difficulty,task_type,failure_type,target_video_id,target_rank,top_1_video_id,moment_hit_at_3,target_window_1_transcript,target_window_2_transcript,target_window_3_transcript
1,R1-9,Đoạn clip là cảnh thu hoạch dứa ở miền Tây: mộ...,medium,KIS,top10_moment_miss,L27_V013,4,L27_V015,False,quý vị ơi bây giờ chúng ta sẽ cùng nhau để đi ...,là là những ụ trời với những người phụ nữ đang...,kh cầu đúc vang danh cho tới nay khóm cầu đúc ...
5,R1-22,Đoạn video về một người phụ nữ dạy nấu ăn cho ...,easy,QA,target_outside_top10,L26_V178,24,L26_V046,True,ăn vào trong chảo cho đầu hành lá vô tôm tươi ...,đây rấtt hấp dẫn và để xem lại công thức của m...,h vô là để cho cái phần lịch xay á nó không có...
7,R2-11,Cảnh quay một người phụ nữ lớn tuổi đang đọc s...,medium,KIS,target_outside_top10,L30_V014,13,L30_V050,True,đ đến ba mươi phần trăm sau cái giá bìa với xu...,không đặt cọc không biên nhận không bắt buộc t...,chào
8,R3-2,Cảnh quay cận cảnh một miếng thiếc có nhiều lỗ...,medium,KIS,target_outside_top10,L29_V020,23,K18_V023,True,miềnây bánh dần vàng không chỉ là món ăn mà cò...,thị trường thì bằng bào này thì thường là ngườ...,và thịt bánh mịn màng hơn bột làm bánh được xa...
9,R3-15,Trong một đoạn video nấu ăn một món ăn về tôm....,medium,KIS,target_outside_top10,L26_V222,81,L26_V227,True,vàúp ngăn ngừa ung thư chống tiểu đường và còn...,có chứa vit ta min xê và pi ta min ca ăn sống ...,thật là mềm và tr chọn với một ít muối tiêu để...
10,R2-1,Nhiều người mặt áo cờ đỏ sao vàng đứng trước b...,easy,KIS,target_outside_top10,K03_V019,512,K14_V013,True,ph phát một ngàn ổ bánh mì miễn phí cho những ...,là nét đẹp văn hóa tinh thần của người dân côn...,rào ngày mười tám tháng mười âm lịch h hàng nă...
11,R2-3,Đoạn clip về một gian trưng bày văn hóa - du l...,medium,QA,target_outside_top10,K17_V003,173,K08_V026,True,đôi tai nhanh nhận này lựa chọn những quả hậu ...,h các dân tộc tham gia các trò chơi dân gian t...,cất cà chua nhầy nhụa trên mặt đất sốt cà chua...
16,R2-8,Đoạn clip mở đầu với hình ảnh cây cầu có hệ th...,medium,KIS,target_outside_top10,K03_V023,488,L22_V015,False,và biến ổ của nó thành h hai ngũ đôngáo đua xe...,thận qua thuận trong gần một tháng bà l để chu...,vì chiều cao và các cành cây cứng cáp đáp ứng ...
18,R3-20,Một cái bàn có để nhiều hộp đĩa CD. Một trong ...,medium,QA,target_outside_top10,K06_V022,45,K02_V016,True,d dài đến phong trào duy tân năm một ngàn tám ...,để khi hát cái bài quang trung trận đạo lên tự...,thư giãn bên nhau đường phố của thành phố mình...
20,R1-7,Đoạn clip cần tìm là cảnh quay trong một khu r...,medium,KIS,target_outside_top10,L29_V023,18,K03_V017,True,coi như là nó nó sẽ phân dùng nhóm chim giường...,hay thế ở thành thị là những chiếc xe đó cho n...,ph phần lớn là cây năng như vậy thì người ta p...


In [12]:
historical_comparison = None

if (HISTORICAL_PATH.is_file()) :
    historical = pd.read_csv(HISTORICAL_PATH)
    selected = historical[
        historical["method_id"].astype(str).str.startswith("R3_bge_v2_m3__S2_first_stage_reranker_50_50")
    ].copy()

    if (len(selected) == 1) :
        old = selected.iloc[0]
        historical_comparison = pd.DataFrame([
            {
                "evaluation" : "Old Stage 7",
                "videos"     : 50,
                "queries"    : 40,
                "Video R@1"  : float(old["video_recall_at_1"]),
                "Video R@5"  : float(old["video_recall_at_5"]),
                "Video MRR"  : float(old["video_mrr"]),
            },
            {
                "evaluation" : "Full production",
                "videos"     : manifest.video_count,
                "queries"    : len(quality_results),
                "Video R@1"  : quality_summary["video_recall_at_1"],
                "Video R@5"  : quality_summary["video_recall_at_5"],
                "Video MRR"  : quality_summary["video_mrr"],
            },
        ])
        print("Historical context")
        display(historical_comparison)
    else :
        print("Historical Stage 7 BGE row was not uniquely identified; skipping the comparison.")
else :
    print("Historical Stage 7 summary not found; skipping the comparison.")

print("\n" + "=" * 72)
print("FULL-CORPUS FINAL SYSTEM VALIDATION")
print("=" * 72)

print("\nCorpus")
print(f"  Videos                  : {manifest.video_count:,}")
print(f"  Searchable windows      : {manifest.eligible_window_count:,}")
print(f"  Queries                 : {len(quality_results):,}")

print("\nVideo retrieval")
print(f"  R@1                     : {quality_summary['video_recall_at_1']:.3f}")
print(f"  R@3                     : {quality_summary['video_recall_at_3']:.3f}")
print(f"  R@5                     : {quality_summary['video_recall_at_5']:.3f}")
print(f"  R@10                    : {quality_summary['video_recall_at_10']:.3f}")
print(f"  R@20                    : {quality_summary['video_recall_at_20']:.3f}")
print(f"  R@50                    : {quality_summary['video_recall_at_50']:.3f}")
print(f"  MRR                     : {quality_summary['video_mrr']:.3f}")
print(f"  Median target rank      : {quality_summary['median_target_rank']:.1f}")

print("\nTemporal support")
print(f"  Moment Hit@1            : {quality_summary['moment_hit_at_1']:.3f}")
print(f"  Moment Hit@3            : {quality_summary['moment_hit_at_3']:.3f}")
print(f"  Video+Moment@10         : {quality_summary['video_plus_moment_at_10']:.3f}")
print(f"  Video+Moment@50         : {quality_summary['video_plus_moment_at_50']:.3f}")

print("\nFailures")
print(f"  Target outside Top 10   : {(quality_results['target_rank'] > 10).sum():,}")
print(f"  Target outside Top 50   : {(quality_results['target_rank'] > 50).sum():,}")

print("\nRuntime")
print(f"  p50                     : {latency_summary['p50_ms']:.1f} ms")
print(f"  p90                     : {latency_summary['p90_ms']:.1f} ms")
print(f"  p95                     : {latency_summary['p95_ms']:.1f} ms")
print(f"  max                     : {latency_summary['max_ms']:.1f} ms")
print(f"  Peak GPU allocated      : {latency_summary['peak_gpu_allocated_gib']:.2f} GiB")

print("\nDeterminism")
print(f"  Queries                 : {latency_summary['deterministic_query_count']} / {latency_summary['deterministic_query_total']}")

print("\nRelease validation       : PASS")
print("Reports                  :", REPORT_ROOT)

engine.close()

Historical context


,evaluation,videos,queries,Video R@1,Video R@5,Video MRR
0,Old Stage 7,50,40,0.750,0.90,0.820524
1,Full production,1478,40,0.425,0.55,0.491090



FULL-CORPUS FINAL SYSTEM VALIDATION

Corpus
  Videos                  : 1,478
  Searchable windows      : 26,117
  Queries                 : 40

Video retrieval
  R@1                     : 0.425
  R@3                     : 0.475
  R@5                     : 0.550
  R@10                    : 0.700
  R@20                    : 0.750
  R@50                    : 0.875
  MRR                     : 0.491
  Median target rank      : 4.0

Temporal support
  Moment Hit@1            : 0.875
  Moment Hit@3            : 0.950
  Video+Moment@10         : 0.675
  Video+Moment@50         : 0.850

Failures
  Target outside Top 10   : 12
  Target outside Top 50   : 5

Runtime
  p50                     : 2330.7 ms
  p90                     : 2788.6 ms
  p95                     : 2885.7 ms
  max                     : 3032.4 ms
  Peak GPU allocated      : 3.49 GiB

Determinism
  Queries                 : 40 / 40

Release validation       : PASS
Reports                  : /content/drive/MyDrive/aic26/asr_mod